# Lending Club Loan Default Risk Analysis
## EDA & Hypothesis Testing

**Dataset:** Lending Club accepted loans (2013–2016), 69,981 cleaned loans (sampled from 1.03M closed loans), 26 columns
**Target:** `default` → 1 = Charged Off, 0 = Fully Paid
**Business Problem:** Which borrower and loan characteristics are associated with default, and how can a lender tighten approval rules or adjust pricing?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st
import statsmodels.api as sm
import statsmodels.stats.weightstats as ssw
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.proportion import proportions_ztest as ppz
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", font_scale=1.1)
PALETTE = {"Fully Paid": "#2ecc71", "Charged Off": "#e74c3c"}
PAL01   = {0: "#2ecc71", 1: "#e74c3c"}
alpha   = 0.05   # significance level for all tests

## 1. Data Loading & Overview

In [2]:
df = pd.read_csv("../data/processed/loans_clean.csv")
df["issue_dt"] = pd.to_datetime(df["issue_dt"])
df["grade"] = pd.Categorical(df["grade"], categories=list("ABCDEFG"), ordered=True)

print(f"Shape: {df.shape}")
print(f"\nDefault rate: {df['default'].mean():.2%}")
print(df.info())

Shape: (69981, 26)

Default rate: 20.22%
<class 'pandas.DataFrame'>
RangeIndex: 69981 entries, 0 to 69980
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   loan_amnt            69981 non-null  float64       
 1   term                 69981 non-null  int64         
 2   int_rate             69981 non-null  float64       
 3   installment          69981 non-null  float64       
 4   grade                69981 non-null  category      
 5   sub_grade            69981 non-null  str           
 6   home_ownership       69981 non-null  str           
 7   annual_inc           69981 non-null  float64       
 8   verification_status  69981 non-null  str           
 9   purpose              69981 non-null  str           
 10  addr_state           69981 non-null  str           
 11  dti                  69981 non-null  float64       
 12  delinq_2yrs          69981 non-null  float64       
 13  o

In [3]:
df.describe().round(2)

,loan_amnt,term,int_rate,installment,annual_inc,dti,delinq_2yrs,open_acc,pub_rec,revol_bal,revol_util,total_acc,issue_dt,default,emp_missing,emp_years,issue_year,fico_avg,annual_inc_capped,log_income
count,69981.00,69981.00,69981.00,69981.00,69981.00,69981.00,69981.00,69981.00,69981.00,69981.00,69981.00,69981.00,69981,69981.0,69981.00,69981.00,69981.00,69981.00,69981.00,69981.00
mean,14535.54,41.79,13.16,441.38,75989.19,18.43,0.33,11.73,0.23,16485.59,53.16,25.27,2015-04-09 16:01:06.669524,0.2,0.06,6.03,2014.81,695.76,74367.77,11.08
min,1000.00,36.00,5.32,30.12,3000.00,0.00,0.00,1.00,0.00,0.00,0.00,2.00,2013-01-01 00:00:00,0.0,0.00,0.00,2013.00,662.00,3000.00,8.01
25%,8000.00,36.00,9.67,255.90,45434.28,12.09,0.00,8.00,0.00,6131.00,35.70,16.00,2014-07-01 00:00:00,0.0,0.00,3.00,2014.00,672.00,45434.28,10.72
50%,12175.00,36.00,12.79,379.16,65000.00,17.93,0.00,11.00,0.00,11310.00,53.60,24.00,2015-06-01 00:00:00,0.0,0.00,6.00,2015.00,687.00,65000.00,11.08
75%,20000.00,36.00,15.88,582.08,90000.00,24.38,0.00,14.00,0.00,20087.00,71.50,32.00,2016-02-01 00:00:00,0.0,0.00,10.00,2016.00,712.00,90000.00,11.41
max,40000.00,60.00,30.99,1536.95,8706582.00,58.72,21.00,90.00,63.00,1746716.00,100.00,119.00,2016-12-01 00:00:00,1.0,1.00,10.00,2016.00,847.50,250000.00,15.98
std,8536.70,10.27,4.61,256.35,73016.13,8.44,0.89,5.48,0.67,23283.31,23.76,11.95,NaN,0.4,0.23,3.58,0.99,30.32,41838.24,0.53


In [4]:
# Missing values
df.isna().mean().sort_values(ascending=False).head(10)

loan_amnt              0.0
term                   0.0
int_rate               0.0
installment            0.0
grade                  0.0
sub_grade              0.0
home_ownership         0.0
annual_inc             0.0
verification_status    0.0
purpose                0.0
dtype: float64

## 2. Target Variable Distribution

In [5]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["default"].value_counts()
bars = ax.bar(["Fully Paid", "Charged Off"], [counts[0], counts[1]],
              color=["#2ecc71", "#e74c3c"], edgecolor="white", width=0.5)
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 400,
            f"{b.get_height():,.0f}", ha="center", fontweight="bold")
ax.set_title("Loan Outcome Distribution", fontsize=14, fontweight="bold")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 3. Univariate Analysis

### 3.1 Numeric Features

In [6]:
num_cols = ["loan_amnt", "int_rate", "annual_inc", "dti", "fico_avg", "revol_util"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), num_cols):
    ax.hist(df[col].dropna(), bins=40, color="#3498db", edgecolor="white", alpha=0.85)
    ax.set_title(col, fontweight="bold")
    ax.set_ylabel("Frequency")
fig.suptitle("Distribution of Key Numeric Features", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### 3.2 Categorical Features

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Grade
df["grade"].value_counts().sort_index().plot.bar(ax=axes[0,0], color="#3498db", edgecolor="white")
axes[0,0].set_title("Loans by Grade", fontweight="bold")
axes[0,0].set_ylabel("Count"); axes[0,0].tick_params(axis="x", rotation=0)

# Term
df["term"].value_counts().sort_index().plot.bar(ax=axes[0,1], color="#9b59b6", edgecolor="white", width=0.4)
axes[0,1].set_title("Loans by Term (months)", fontweight="bold")
axes[0,1].set_ylabel("Count"); axes[0,1].tick_params(axis="x", rotation=0)

# Home ownership
order_ho = df["home_ownership"].value_counts().index
sns.countplot(data=df, x="home_ownership", order=order_ho, color="#e67e22", ax=axes[1,0])
axes[1,0].set_title("Loans by Home Ownership", fontweight="bold")
axes[1,0].tick_params(axis="x", rotation=30)

# Purpose top 8
top_purposes = df["purpose"].value_counts().nlargest(8).index
sns.countplot(data=df[df["purpose"].isin(top_purposes)], y="purpose",
              order=top_purposes, color="#1abc9c", ax=axes[1,1])
axes[1,1].set_title("Top 8 Loan Purposes", fontweight="bold")

plt.tight_layout()
plt.show()

## 4. Bivariate Analysis

### 4.1 Key Features: Fully Paid vs Charged Off

In [8]:
bi_cols = ["int_rate", "fico_avg", "annual_inc", "dti"]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, col in zip(axes, bi_cols):
    sns.boxplot(data=df, x="default", y=col, ax=ax, width=0.45, fliersize=2)
    ax.set_xticklabels(["Fully Paid", "Charged Off"])
    ax.set_title(col, fontweight="bold")
fig.suptitle("Key Features: Fully Paid vs Charged Off", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 4.2 Default Rate by Grade

In [9]:
fig, ax = plt.subplots(figsize=(8, 5))
grade_def = df.groupby("grade", observed=True)["default"].mean() * 100
bars = ax.bar(grade_def.index.astype(str), grade_def.values,
              color=sns.color_palette("YlOrRd", n_colors=7), edgecolor="white")
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.5,
            f"{b.get_height():.1f}%", ha="center", fontweight="bold", fontsize=10)
ax.set_title("Default Rate by Loan Grade", fontsize=14, fontweight="bold")
ax.set_ylabel("Default Rate (%)")
ax.set_xlabel("Grade")
plt.tight_layout()
plt.show()

### 4.3 Default Rate by Purpose

In [10]:
fig, ax = plt.subplots(figsize=(9, 5))
purp_def = (df[df["purpose"].isin(top_purposes)]
            .groupby("purpose")["default"].mean().sort_values(ascending=True) * 100)
purp_def.plot.barh(ax=ax, color="#e74c3c", edgecolor="white")
for i, v in enumerate(purp_def.values):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center", fontweight="bold")
ax.set_title("Default Rate by Loan Purpose", fontsize=14, fontweight="bold")
ax.set_xlabel("Default Rate (%)")
plt.tight_layout()
plt.show()

### 4.4 Default Rate by Term

In [11]:
fig, ax = plt.subplots(figsize=(6, 4))
term_def = df.groupby("term")["default"].mean() * 100
bars = ax.bar(term_def.index.astype(str), term_def.values,
              color=["#3498db", "#e74c3c"], edgecolor="white", width=0.4)
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.4,
            f"{b.get_height():.1f}%", ha="center", fontweight="bold")
ax.set_title("Default Rate by Loan Term", fontsize=14, fontweight="bold")
ax.set_ylabel("Default Rate (%)")
ax.set_xlabel("Term (months)")
plt.tight_layout()
plt.show()

### 4.5 Default Rate by Verification Status

In [12]:
fig, ax = plt.subplots(figsize=(7, 4))
ver_def = df.groupby("verification_status")["default"].mean().sort_values() * 100
ver_def.plot.bar(ax=ax, color=["#2ecc71", "#f39c12", "#e74c3c"], edgecolor="white", width=0.5)
for i, (idx, v) in enumerate(ver_def.items()):
    ax.text(i, v + 0.3, f"{v:.1f}%", ha="center", fontweight="bold")
ax.set_title("Default Rate by Verification Status", fontsize=14, fontweight="bold")
ax.set_ylabel("Default Rate (%)")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

### 4.6 FICO Score Distribution by Outcome

In [13]:
fig, ax = plt.subplots(figsize=(8, 4))
for label, color in [(0, "#2ecc71"), (1, "#e74c3c")]:
    subset = df[df["default"] == label]["fico_avg"]
    ax.hist(subset, bins=50, density=True, alpha=0.45, color=color,
            label="Fully Paid" if label == 0 else "Charged Off", edgecolor="white")
    subset.plot.kde(ax=ax, color=color, linewidth=2)
ax.set_title("FICO Score Distribution by Loan Outcome", fontsize=14, fontweight="bold")
ax.set_xlabel("FICO Average Score")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Multivariate Analysis

### 5.1 Correlation Heatmap

In [14]:
num_for_corr = ["loan_amnt", "int_rate", "installment", "annual_inc", "dti",
                "fico_avg", "revol_util", "revol_bal", "open_acc", "total_acc",
                "emp_years", "delinq_2yrs", "pub_rec", "default"]
corr = df[num_for_corr].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool)) 
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            annot_kws={"size": 8})
ax.set_title("Correlation Heatmap of Numeric Features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 5.2 Default Rate by Grade × Term

In [15]:
fig, ax = plt.subplots(figsize=(9, 5))
pivot = df.pivot_table(values="default", index="grade", columns="term",
                       aggfunc="mean", observed=True) * 100
pivot.plot.bar(ax=ax, color=["#3498db", "#e74c3c"], edgecolor="white", width=0.7)
ax.set_title("Default Rate by Grade × Term", fontsize=14, fontweight="bold")
ax.set_ylabel("Default Rate (%)")
ax.legend(title="Term (months)")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

### 5.3 Interest Rate by Grade

In [16]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x="grade", y="int_rate",
            palette="YlOrRd", ax=ax, width=0.55, fliersize=2)
ax.set_title("Interest Rate Distribution by Loan Grade", fontsize=14, fontweight="bold")
ax.set_ylabel("Interest Rate (%)")
ax.set_xlabel("Grade")
plt.tight_layout()
plt.show()

## 6. Hypothesis Tests

All tests use significance level **α = 0.05**.

> **Note on test selection:** We use 4 different tests from our inferential statistics toolkit to answer 4 distinct business questions. Each test is chosen based on the **type of data** (continuous vs categorical) and the **number of groups** being compared.

### Test 1: Two-Sample t-test — Do defaulters earn less/more than non-defaulters?

**Test used:** Two-sample t-test for independent samples (`ssw.ttest_ind`)
— compares means of **two independent, unrelated groups** (Test #3 from our notes)

---

#### 🏦 Business Logic

| Item | Detail |
|------|--------|
| **Feature** | `annual_inc` (Annual Income in USD) |
| **Groups** | Group 1: Fully Paid borrowers (`default = 0`) — Group 2: Charged Off borrowers (`default = 1`) |
| **What it represents** | A borrower's self-reported yearly income at the time of loan application |
| **Why it matters for a lender** | If defaulters have significantly lower income, the lender can set **minimum income thresholds** for loan approval, or require **additional collateral** from low-income applicants |
| **Business decision** | Should the bank use annual income as a **screening criterion** during loan approval? |

---

- **H₀:** Mean annual income of Fully Paid = Mean annual income of Charged Off (μ₁ − μ₂ = 0)
- **H₁:** Mean annual income of Fully Paid ≠ Mean annual income of Charged Off (μ₁ − μ₂ ≠ 0)
- **Why this test?** We have two independent groups (paid vs defaulted) and a continuous variable (income). Sample sizes are large and unequal, making the two-sample t-test appropriate.

In [17]:
# ── Test 1: Two-sample t-test (ssw.ttest_ind) ──
# Comparing annual_inc between Fully Paid and Charged Off borrowers

paid_inc    = df[df["default"] == 0]["annual_inc"]   # income of fully paid borrowers
default_inc = df[df["default"] == 1]["annual_inc"]   # income of defaulted borrowers

print("=" * 60)
print("TEST 1: Two-Sample t-test — Annual Income vs Default")
print("=" * 60)
print(f"\nFully Paid  — n = {len(paid_inc):,},  mean = ${paid_inc.mean():,.0f},  median = ${paid_inc.median():,.0f}")
print(f"Charged Off — n = {len(default_inc):,},  mean = ${default_inc.mean():,.0f},  median = ${default_inc.median():,.0f}")

# H0: mu_paid - mu_default = 0 (no difference)
# H1: mu_paid - mu_default != 0 (two-tailed)
t_stat, p_val, dof = ssw.ttest_ind(paid_inc, default_inc, value=0, alternative='two-sided')

print(f"\nt-statistic = {t_stat:.4f}")
print(f"p-value     = {p_val:.2e}")
print(f"df          = {dof:.1f}")
print(f"\nDecision (α = {alpha}): {'✅ Reject H₀' if p_val < alpha else '❌ Fail to reject H₀'}")
# effect size: with ~70K rows even tiny gaps are "significant", so also report Cohen's d
sp = np.sqrt(((len(paid_inc)-1)*paid_inc.var() + (len(default_inc)-1)*default_inc.var()) / (len(paid_inc)+len(default_inc)-2))
cohen_d = (paid_inc.mean() - default_inc.mean()) / sp
print(f"Cohen's d   = {cohen_d:.3f}  (|d| < 0.2 = negligible)")
# robustness: income is heavily right-skewed (max $8.7M), so also run Welch's t-test and Mann-Whitney
print(f"Welch t     = {st.ttest_ind(paid_inc, default_inc, equal_var=False).statistic:.2f}   Mann-Whitney p = {st.mannwhitneyu(paid_inc, default_inc).pvalue:.1e}")
print(f"\n💡 Business Insight: Fully-paid borrowers earn significantly more on average, but the effect is negligible")
print("   (about $6.8K, d ≈ 0.09). Income alone is a weak screening criterion; use it together with DTI and FICO.")

TEST 1: Two-Sample t-test — Annual Income vs Default

Fully Paid  — n = 55,828,  mean = $77,355,  median = $65,000
Charged Off — n = 14,153,  mean = $70,601,  median = $60,000

t-statistic = 9.8365
p-value     = 8.11e-23
df          = 69979.0

Decision (α = 0.05): ✅ Reject H₀
Cohen's d   = 0.093  (|d| < 0.2 = negligible)
Welch t     = 7.15   Mann-Whitney p = 2.4e-76

💡 Business Insight: Fully-paid borrowers earn significantly more on average, but the effect is negligible
   (about $6.8K, d ≈ 0.09). Income alone is a weak screening criterion; use it together with DTI and FICO.


### Test 2: Chi-Square Test of Independence — Does default rate depend on loan purpose?

**Test used:** Chi-square test of independence with Cramér's V (`st.chi2_contingency`)
— tests whether **two categorical variables are associated** using a **contingency table** (Test #12 from our notes)

---

#### 🏦 Business Logic

| Item | Detail |
|------|--------|
| **Feature 1** | `purpose` (Reason for taking the loan — e.g., debt_consolidation, small_business, home_improvement, credit_card, etc.) |
| **Feature 2** | `default` (Loan outcome — 0 = Fully Paid, 1 = Charged Off) |
| **What it represents** | Whether borrowers who take loans for different purposes have different default rates |
| **Why it matters for a lender** | If certain purposes (like small_business) show significantly higher default rates, the lender can **add extra scrutiny**, **require collateral**, or **charge higher interest** for those categories |
| **Business decision** | Should the bank treat all loan purposes equally, or apply **purpose-specific risk policies**? |
| **Cramér's V** | Tells us not just *whether* an association exists (p-value), but **how strong** that association is (0 = no association, 1 = perfect association) |

---

- **H₀:** Default status and loan purpose are **independent** (no association)
- **H₁:** Default status and loan purpose are **associated** (not independent)
- **Why this test?** Both variables are categorical. We build a contingency table (purpose × default) and check if the observed counts differ significantly from what we'd expect if purpose and default were unrelated.

In [18]:
# ── Test 2: Chi-Square Test of Independence (st.chi2_contingency) ──
# Testing association between purpose (categorical) and default (categorical)

print("=" * 60)
print("TEST 2: Chi-Square Independence — Loan Purpose vs Default")
print("=" * 60)

# Build contingency table: rows = purpose, columns = default (0, 1)
ct_purpose = pd.crosstab(df["purpose"], df["default"])
print("\nContingency Table (showing top 5 purposes by volume):")
print(ct_purpose.loc[df["purpose"].value_counts().nlargest(5).index])

# Run chi-square test of independence
chi2_stat, p_val, dof, expected = st.chi2_contingency(ct_purpose)

# Cramér's V — measures strength of association (0 to 1)
n = ct_purpose.values.sum()
V = np.sqrt(chi2_stat / (n * (min(ct_purpose.shape) - 1)))

print(f"\nχ² statistic = {chi2_stat:.4f}")
print(f"dof          = {dof}")
print(f"p-value      = {p_val:.2e}")
print(f"Cramér's V   = {V:.4f}  (strength of association)")
print(f"\nDecision (α = {alpha}): {'✅ Reject H₀' if p_val < alpha else '❌ Fail to reject H₀'}")

# Show default rate by purpose
print("\nDefault rate by purpose (sorted):")
purpose_default = df.groupby("purpose")["default"].mean().sort_values(ascending=False) * 100
for purp, rate in purpose_default.items():
    print(f"  {purp:25s}  {rate:5.1f}%")

print(f"\n💡 Business Insight: Loan purpose is significantly associated with default risk (V = {V:.3f}).")
big = df["purpose"].value_counts(); big = big[big >= 300].index
top = purpose_default[purpose_default.index.isin(big)]
print(f"   → Among purposes with at least 300 loans, {top.index[0]} is riskiest ({top.iloc[0]:.1f}%) and {top.index[-1]} safest ({top.iloc[-1]:.1f}%).")
print("   → renewable_energy (n=37) and wedding (n=49) are too small to rank reliably.")
print("   → The bank should require additional documentation or collateral for high-risk purposes.")

TEST 2: Chi-Square Independence — Loan Purpose vs Default

Contingency Table (showing top 5 purposes by volume):
default                 0     1
purpose                        
debt_consolidation  32522  8870
credit_card         13382  2744
home_improvement     3538   755
other                2773   786
major_purchase       1014   273

χ² statistic = 226.9483
dof          = 12
p-value      = 8.58e-42
Cramér's V   = 0.0569  (strength of association)

Decision (α = 0.05): ✅ Reject H₀

Default rate by purpose (sorted):
  renewable_energy            35.1%
  small_business              29.4%
  house                       23.6%
  moving                      22.3%
  other                       22.1%
  debt_consolidation          21.4%
  major_purchase              21.2%
  medical                     20.9%
  vacation                    20.4%
  home_improvement            17.6%
  credit_card                 17.0%
  car                         15.2%
  wedding                      6.1%

💡 Busines

### Test 3: One-Way ANOVA + Tukey HSD — Does interest rate differ across loan grades?

**Test used:** One-way ANOVA (`ols` + `anova_lm`) with Tukey HSD post-hoc (`pairwise_tukeyhsd`)
— compares means of **3+ groups** by **one factor** using the **F-test**, then Tukey tells us **which pairs differ** (Test #5 from our notes)

---

#### 🏦 Business Logic

| Item | Detail |
|------|--------|
| **Continuous variable (Y)** | `int_rate` (Interest Rate %, e.g., 10.5%, 15.3%) — the price the borrower pays for the loan |
| **Grouping factor (X)** | `grade` (Loan Grade: A, B, C, D, E, F, G) — LendingClub's internal credit-risk rating, where A = safest, G = riskiest |
| **What it represents** | Whether LendingClub charges different interest rates based on the borrower's risk grade |
| **Why it matters** | This validates the bank's **risk-based pricing model** — if interest rates are statistically different across grades, the pricing tiers are working correctly. If not, the bank is under-charging risky borrowers or over-charging safe ones |
| **Business decision** | Is the bank's pricing strategy (charge more for riskier grades) **statistically sound**, and are the tiers **distinct enough**? |
| **Tukey HSD** | ANOVA only says "at least one grade differs." Tukey tells us **which specific grade-pairs** have significantly different rates (e.g., is Grade C significantly different from Grade D?) |

---

- **H₀:** Mean interest rate is **equal** across all loan grades A–G (μA = μB = μC = μD = μE = μF = μG)
- **H₁:** **At least one** grade has a different mean interest rate
- **Why this test?** We have one continuous variable (interest rate) and one categorical factor with 7 levels (grades A–G). ANOVA checks if at least one group mean differs; Tukey post-hoc identifies which specific pairs differ.

In [19]:
# ── Test 3: One-Way ANOVA + Tukey HSD (ols + anova_lm + pairwise_tukeyhsd) ──
# Comparing int_rate across 7 grade groups (A through G)

print("=" * 60)
print("TEST 3: One-Way ANOVA + Tukey — Interest Rate vs Grade")
print("=" * 60)

# Mean interest rate per grade
print("\nMean interest rate per grade:")
for g, grp in df.groupby("grade", observed=True):
    print(f"  Grade {g}: {grp['int_rate'].mean():.2f}%  (n = {len(grp):,})")

# One-way ANOVA using OLS model (as per our notes: ols + anova_lm)
df["grade_str"] = df["grade"].astype(str)   # ols needs string, not Categorical
model = ols('int_rate ~ grade_str', data=df).fit()
anova_table = sm.stats.anova_lm(model)

print(f"\nANOVA Table:")
print(anova_table.round(4))
print(f"\nF-statistic = {anova_table.loc['grade_str', 'F']:.4f}")
print(f"p-value     = {anova_table.loc['grade_str', 'PR(>F)']:.2e}")

p_val = anova_table.loc['grade_str', 'PR(>F)']
print(f"\nDecision (α = {alpha}): {'✅ Reject H₀' if p_val < alpha else '❌ Fail to reject H₀'}")
print("→ At least one grade has a significantly different mean interest rate.")

# Tukey HSD post-hoc — which specific grade pairs differ?
print("\n" + "=" * 60)
print("Tukey HSD Post-Hoc Test (which grade pairs differ?):")
print("=" * 60)
tukey = pairwise_tukeyhsd(df["int_rate"], df["grade_str"])
print(tukey)

eta2 = anova_table.loc['grade_str', 'sum_sq'] / anova_table['sum_sq'].sum()
print(f"\neta-squared = {eta2:.2f}")
print(f"\n💡 Business Insight: Interest rates are significantly different across ALL grades.")
print("   → Caution: LendingClub assigns the grade FROM the interest rate, so this test is close to a tautology.")
print("   → It shows the tiers are priced distinctly. Whether prices match risk is shown by default rates rising with grade (section 4.2).")

TEST 3: One-Way ANOVA + Tukey — Interest Rate vs Grade

Mean interest rate per grade:
  Grade A: 7.11%  (n = 11,959)
  Grade B: 10.58%  (n = 20,177)
  Grade C: 13.92%  (n = 19,955)
  Grade D: 17.47%  (n = 10,636)
  Grade E: 20.62%  (n = 5,100)
  Grade F: 24.41%  (n = 1,728)
  Grade G: 27.05%  (n = 426)



ANOVA Table:
                df        sum_sq      mean_sq            F  PR(>F)
grade_str      6.0  1.366246e+06  227707.7013  130166.2124     0.0
Residual   69974.0  1.224098e+05       1.7494          NaN     NaN

F-statistic = 130166.2124
p-value     = 0.00e+00

Decision (α = 0.05): ✅ Reject H₀
→ At least one grade has a significantly different mean interest rate.

Tukey HSD Post-Hoc Test (which grade pairs differ?):


Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj  lower   upper  reject
---------------------------------------------------
     A      B   3.4675   0.0  3.4225  3.5125   True
     A      C   6.8141   0.0   6.769  6.8592   True
     A      D  10.3584   0.0 10.3065 10.4104   True
     A      E  13.5105   0.0 13.4452 13.5757   True
     A      F  17.3035   0.0 17.2031 17.4038   True
     A      G  19.9385   0.0 19.7463 20.1308   True
     B      C   3.3466   0.0  3.3077  3.3855   True
     B      D    6.891   0.0  6.8442  6.9377   True
     B      E   10.043   0.0  9.9819 10.1041   True
     B      F   13.836   0.0 13.7383 13.9338   True
     B      G  16.4711   0.0 16.2801  16.662   True
     C      D   3.5444   0.0  3.4976  3.5912   True
     C      E   6.6964   0.0  6.6352  6.7576   True
     C      F  10.4894   0.0 10.3916 10.5872   True
     C      G  13.1245   0.0 12.9335 13.3154   True
     D      E    3.152   0.0  3.0856  3.2184   True
     D      

### Test 4: Two-Proportion z-test — Do 60-month loans default more than 36-month loans?

**Test used:** Two-proportion z-test (`ppz` / `proportions_ztest`)
— compares the **default proportions of two independent groups** (Test #10 from our notes)

---

#### 🏦 Business Logic

| Item | Detail |
|------|--------|
| **Feature** | `term` (Loan Duration — either 36 months or 60 months) |
| **Group 1** | 60-month term borrowers — longer commitment, higher total interest paid |
| **Group 2** | 36-month term borrowers — shorter commitment, lower total interest paid |
| **What we measure** | The **proportion (percentage) of defaults** in each group = (number of defaults ÷ total loans in that group) |
| **What it represents** | Whether longer-term loans carry a significantly higher default **rate** |
| **Why it matters for a lender** | If 60-month loans default at a significantly higher rate, the bank should either **limit 60-month offerings**, **charge a premium**, or **tighten approval criteria** for longer terms |
| **Business decision** | Should the bank apply **stricter rules for 60-month loans** compared to 36-month loans? |

---

- **H₀:** Default proportion of 60-month loans <= Default proportion of 36-month loans (p₆₀ − p₃₆ <= 0)
- **H₁:** Default proportion of 60-month loans > Default proportion of 36-month loans (p₆₀ − p₃₆ > 0) → **right-tailed test**
- **Why this test?** We're comparing proportions (default rate = count/total) of two independent groups (36-month vs 60-month borrowers). This is exactly the two-proportion z-test scenario.

In [20]:
# ── Test 4: Two-Proportion z-test (ppz / proportions_ztest) ──
# Comparing default PROPORTIONS between 60-month and 36-month loans

print("=" * 60)
print("TEST 4: Two-Proportion z-test — Default Rate by Term")
print("=" * 60)

# Count defaults and total loans per term
term_60 = df[df["term"] == 60]
term_36 = df[df["term"] == 36]

defaults_60 = term_60["default"].sum()       # count of defaults in 60-month
defaults_36 = term_36["default"].sum()       # count of defaults in 36-month
n_60 = len(term_60)                          # total 60-month loans
n_36 = len(term_36)                          # total 36-month loans

prop_60 = defaults_60 / n_60                 # default proportion 60-month
prop_36 = defaults_36 / n_36                 # default proportion 36-month

print(f"\n60-month loans: {defaults_60:,} defaults out of {n_60:,}  →  default rate = {prop_60:.1%}")
print(f"36-month loans: {defaults_36:,} defaults out of {n_36:,}  →  default rate = {prop_36:.1%}")
print(f"Difference:     {prop_60 - prop_36:.1%} percentage points")

# H0: p_60 - p_36 = 0 (equal default rates)
# H1: p_60 - p_36 > 0 (60-month defaults MORE) → right tail → 'larger'
# ppz(counts_array, nobs_array, value=claimed_difference, alternative=)
# NOTE: 60-month goes FIRST because H1 says 60-month > 36-month
counts = np.array([defaults_60, defaults_36])
nobs   = np.array([n_60, n_36])

z_stat, p_val = ppz(counts, nobs, value=0, alternative='larger')

print(f"\nz-statistic = {z_stat:.4f}")
print(f"p-value     = {p_val:.2e}")
print(f"\nDecision (α = {alpha}): {'✅ Reject H₀' if p_val < alpha else '❌ Fail to reject H₀'}")
print(f"\n💡 Business Insight: 60-month loans default at a significantly higher rate (~{prop_60:.0%})")
print(f"   compared to 36-month loans (~{prop_36:.0%}).")
# This test ignores grade. 60-month loans skew to weaker grades, so also compare within each grade.
cell = df.groupby(["grade", "term"], observed=True)["default"].agg(["sum", "size"]).unstack("term")
ratios = (cell[("sum", 60)] / cell[("size", 60)]) / (cell[("sum", 36)] / cell[("size", 36)])
num = den = 0.0
for _, c in cell.iterrows():
    a6, n6, a3, n3 = c[("sum", 60)], c[("size", 60)], c[("sum", 36)], c[("size", 36)]
    num += a6 * (n3 - a3) / (n6 + n3); den += (n6 - a6) * a3 / (n6 + n3)
print(f"\nWithin the same grade, 60-month default is {ratios.min():.1f}-{ratios.max():.1f}x the 36-month rate; Mantel-Haenszel grade-adjusted odds ratio = {num/den:.2f}")
print("   → The raw ~2x gap is inflated by grade mix, but 60-month loans are still riskier within every grade.")
print("   → The bank should tighten approval criteria for 60-month loans,")
print("   especially for grades C–E, where the within-grade gap is largest.")
print("   → Consider offering 36-month alternatives or requiring higher credit scores for 60-month terms.")

TEST 4: Two-Proportion z-test — Default Rate by Term

60-month loans: 5,719 defaults out of 16,875  →  default rate = 33.9%
36-month loans: 8,434 defaults out of 53,106  →  default rate = 15.9%
Difference:     18.0% percentage points

z-statistic = 50.7366
p-value     = 0.00e+00

Decision (α = 0.05): ✅ Reject H₀

💡 Business Insight: 60-month loans default at a significantly higher rate (~34%)
   compared to 36-month loans (~16%).

Within the same grade, 60-month default is 1.1-1.8x the 36-month rate; Mantel-Haenszel grade-adjusted odds ratio = 1.60
   → The raw ~2x gap is inflated by grade mix, but 60-month loans are still riskier within every grade.
   → The bank should tighten approval criteria for 60-month loans,
   especially for grades C–E, where the within-grade gap is largest.
   → Consider offering 36-month alternatives or requiring higher credit scores for 60-month terms.


## 7. Summary of Hypothesis Test Results

| # | Business Question | Test Used | Features / Columns | Statistic | Decision |
|---|---|---|---|---|---|
| 1 | Do defaulters earn less/more? | Two-sample t-test (`ssw.ttest_ind`) | `annual_inc` split by `default` (0/1) | t-stat, p-value | Reject H₀ |
| 2 | Does default depend on loan purpose? | Chi-square independence + Cramér's V (`st.chi2_contingency`) | `purpose` × `default` contingency table | χ², Cramér's V | Reject H₀ |
| 3 | Are interest rates different across grades? | One-way ANOVA + Tukey (`ols` + `anova_lm` + `pairwise_tukeyhsd`) | `int_rate` grouped by `grade` (A–G) | F-stat, Tukey pairs | Reject H₀ |
| 4 | Do 60-month loans default more? | Two-proportion z-test (`ppz`) | Default counts & totals for each `term` (36/60) | z-stat, p-value | Reject H₀ |

**All four null hypotheses were rejected at α = 0.05.**

### Why these 4 tests give the best business coverage:

1. **t-test** → answers whether **income** (a continuous variable) can predict default → helps set **income-based approval thresholds**
2. **Chi-square** → answers whether **loan purpose** (a categorical variable) is linked to default → helps with **purpose-based risk policies**
3. **ANOVA + Tukey** → shows the **pricing tiers** (7 grade levels) are statistically distinct (near-tautological, since grade is assigned from the rate)
4. **Proportion z-test** → compares **default rates** (proportions) between two term groups → helps decide **term-specific lending policies**

## 8. Key Business Insights

1. **Grade is the strongest default predictor** — default rate climbs from 6.3% (Grade A) to 50.7% (Grade G).
2. **60-month loans are riskier** — 33.9% vs 15.9% overall, but 60-month loans skew to weaker grades. Within the same grade the gap is about 1.1–1.8× (grade-adjusted odds ratio ≈ 1.6), not 2×.
3. **Higher interest rate ↔ higher default** — but this is partly circular (riskier borrowers receive higher rates). ANOVA confirms the pricing tiers are statistically distinct.
4. **FICO score inversely correlates with default** — lower FICO borrowers default more frequently.
5. **Small business loans carry the highest default rate** among purposes with at least 300 loans (29.4%). Tiny categories such as renewable energy (n=37) and wedding (n=49) are too small to rank.
6. **Income verification does not reduce default** — Verified borrowers actually default slightly more, likely because verification is triggered for riskier applicants.

## 9. Recommendations

1. **Tighten approval for Grade E–G, 60-month loans** — these carry the highest combined risk.
2. **Use income only alongside other signals** — defaulters earn less on average, but the effect is negligible (Cohen's d ≈ 0.09), so income alone is a weak cut-off.
3. **Require additional scrutiny for small-business-purpose loans** — collateral or business plan review.
4. **Supplement income verification** with debt-to-income and credit-utilisation checks.
5. **Limit or add premium pricing for 60-month terms** — they default more even within the same grade (about 1.4× for grades C–E), though the raw ~2× gap overstates it.
6. **Use FICO + Grade + Term + Purpose** as a combined risk-scoring framework for approval decisions.